# 🔍 ADC Survey Filter
Filtrage des ADC ISSCC/VLSI selon des critères de performance personnalisés.

In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Chargement des deux onglets ──────────────────────────────────────────────
FILE = 'ADCsurvey_latest.xlsx'

df_isscc = pd.read_excel(FILE, sheet_name='ISSCC', header=0)
df_vlsi  = pd.read_excel(FILE, sheet_name='VLSI',  header=0)

df_isscc['SOURCE'] = 'ISSCC'
df_vlsi['SOURCE']  = 'VLSI'

df_all = pd.concat([df_isscc, df_vlsi], ignore_index=True)
print(f'Total ADCs chargés : {len(df_all)}  (ISSCC: {len(df_isscc)}, VLSI: {len(df_vlsi)})')

Total ADCs chargés : 752  (ISSCC: 459, VLSI: 293)


## 📋 Colonnes disponibles et plages de valeurs

In [8]:
# Colonnes numériques de performance
PERF_COLS = [
    'SNDR_plot [dB]', 'SNDR_lf [dB]', 'SNDR_hf [dB]', 'SNR [dB]', 'DR [dB]',
    '-THD [dB]', 'SFDR [dB]', 'fsnyq [Hz]', 'fs [Hz]', 'fin_hf [Hz]',
    'P [W]', 'AREA [mm^2]', 'Csamp [pF]', 'OSR',
    'FOMW_lf [fJ/conv-step]', 'FOMW_hf [fJ/conv-step]',
    'FOMS_lf [dB]', 'FOMS_hf [dB]', 'P/fsnyq [pJ]',
    'VSUP1 [V]', 'VSUP2 [V]',
]

stats = df_all[PERF_COLS].describe().loc[['min','max','50%']].T
stats.columns = ['Min', 'Max', 'Median']
#

---
## ⚙️ Définir les critères de filtrage

Modifiez le dictionnaire `CRITERIA` ci-dessous.  
Chaque entrée est une **colonne** associée à un tuple `(opérateur, valeur)`.  

**Opérateurs supportés :** `'>'`, `'>='`, `'<'`, `'<='`, `'=='`  
Laissez la liste vide `[]` pour ne poser aucun filtre.

### Exemples rapides :
| Critère | Code |
|---------|------|
| SNDR > 80 dB | `'SNDR_plot [dB]': [('>', 80)]` |
| fsnyq > 1 MHz | `'fsnyq [Hz]': [('>', 1e6)]` |
| Puissance < 10 mW | `'P [W]': [('<', 10e-3)]` |
| FOM Walden < 100 fJ/c-s | `'FOMW_hf [fJ/conv-step]': [('<', 100)]` |
| Fenêtre SNDR : 70–90 dB | `'SNDR_plot [dB]': [('>=', 70), ('<=', 90)]` |

In [9]:
# ══════════════════════════════════════════════════════════
#  🎛️  MODIFIEZ ICI VOS CRITÈRES
# ══════════════════════════════════════════════════════════

CRITERIA = {
    'SNDR_plot [dB]'         : [('>=', 80)],
    'fsnyq [Hz]'             : [('>=', 0.1e6)],
    'P [W]'                : [('<', 20e-3)],
    # 'FOMW_hf [fJ/conv-step]' : [('<', 100)],
    # 'AREA [mm^2]'          : [('<', 1)],
}

# Quelle(s) source(s) chercher ?  'ISSCC', 'VLSI', ou les deux
SOURCES = ['ISSCC', 'VLSI']

# Colonnes à afficher dans le résultat
DISPLAY_COLS = [
    'SOURCE', 'YEAR', 'ARCHITECTURE', 'TECHNOLOGY',
    'SNDR_plot [dB]', 'fsnyq [Hz]', 'P [W]',
    'AREA [mm^2]', 'TITLE', 'ABSTRACT'
]

# ══════════════════════════════════════════════════════════

## 🚀 Application des filtres

In [10]:
OPS = {'>': np.greater, '>=': np.greater_equal,
       '<': np.less,    '<=': np.less_equal, '==': np.equal}

def apply_criteria(df, criteria, sources):
    mask = df['SOURCE'].isin(sources)
    for col, conditions in criteria.items():
        if col not in df.columns:
            print(f'⚠️  Colonne inconnue ignorée : "{col}"')
            continue
        for (op, val) in conditions:
            mask &= OPS[op](df[col], val) & df[col].notna()
    return df[mask].copy()

results = apply_criteria(df_all, CRITERIA, SOURCES)

print(f'\n✅  {len(results)} ADC(s) correspondent aux critères :')
for col, conds in CRITERIA.items():
    for op, val in conds:
        print(f'   • {col} {op} {val:g}')
print(f'   • Source(s) : {", ".join(SOURCES)}')


✅  47 ADC(s) correspondent aux critères :
   • SNDR_plot [dB] >= 80
   • fsnyq [Hz] >= 100000
   • P [W] < 0.02
   • Source(s) : ISSCC, VLSI


In [11]:
# Affichage tabulaire, trié par SNDR décroissant
show_cols = [c for c in DISPLAY_COLS if c in results.columns]

sort_key = 'SNDR_plot [dB]' if 'SNDR_plot [dB]' in results.columns else 'YEAR'
display_df = results[show_cols].sort_values(sort_key, ascending=True).reset_index(drop=True)

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.float_format', lambda x: f'{x:.3g}')
display_df

,SOURCE,YEAR,ARCHITECTURE,TECHNOLOGY,SNDR_plot [dB],fsnyq [Hz],P [W],AREA [mm^2],TITLE,ABSTRACT
0,VLSI,2017,SAR,0.04,80,5.25e+05,0.000143,0.04,A 13b-ENOB 173dB-FoM 2nd -Order NS SAR ADC with Passive Integrators,"This paper presents a low-power 2nd-order noise-shaping (NS) SAR ADC. Instead of using power-hungry op-amps, it uses switches and capacitors to make passive integrators for noise shaping. The over..."
1,VLSI,2025,"SAR, Pipe, Incremental",0.028,80.1,1e+07,0.00185,0.052,An NS-SAR Quantizer-Based Pipeline Incremental Delta-Sigma ADC Using a Current-Regulated Floating Ring Amplifier and Two-Phase Miller Negative-C,"A pipeline incremental ADC (IADC) leverages a 6-bit noise-shaping SAR quantizer for high SQNR at low OSR. A pipeline IADC structure enables continuous sampling and concurrent stage processing, imp..."
2,ISSCC,2017,SDCT,0.13,80.4,3e+07,0.0114,0.17,An 11.4mW 80.4dB-SNDR 15MHz-BW CT Delta-Sigma Modulator Using 6b Double-Noise-Shaped Quantizer,"In Paper 28.2, the University of Florida presents a 4th-order continuous-time delta-sigma modulator, employing a double noise-shaping quantizer, which incorporates both noise-shaped integrating an..."
3,VLSI,2015,SDCT,0.028,80.5,1e+07,0.00316,0.066,"A 13-ENOB, 5 MHz BW, 3.16 mW Multi-Bit Continuous-Time ΔΣ ADC in 28 nm CMOS with Excess-Loop-Delay Compensation Embedded in SAR Quantizer","A 13-ENOB, 5 MHz BW, 3.16 mW 3-bit continuous-time ΔΣ ADC sampling at 432 MHz is presented. For power efficiency, this design utilizes a hybrid feedback feed-forward loop topology with SAR quantiz..."
4,ISSCC,2021,SDCT,0.04,80.9,2.5e+07,0.0037,0.057,A 3.7mW 12.5MHz 81dB-SNDR 4th-Order CTDSM with Single-OTA and 2nd-Order NS-SAR,"In Paper 10.4, University of Texas at Austin presents a 4th-order CT ΔΣ modulator with single-OTA and 2nd-order NS-SAR that achieves 81dB-SNDR in 12.5MHz bandwidth and consumes 3.7mW."
5,ISSCC,2008,"SDSC, TI",0.18,81,5e+06,0.015,3.67,"A Noise-Coupled Time-Interleaved delta sigma AD with 4.2MHz Bw, -98dB THD, and 79dB SNDR","In this paper, two prototype versions of a SC time-interleaved DeltaSigma ADC are described. Both use quantization- noise coupling for enhanced noise shaping. They achieve high linearity, and FOMs..."
6,VLSI,2022,SDCT,0.028,81.6,3.12e+07,0.0064,0.072,An 81.6dB SNDR 15.625MHz BW 3rd Order CT SDM with a True TI NS Quantizer,This paper presents a CT SDM with a TINSQTZ. Complete parallelization of the NSQTZ operations relaxes loop filtering and residue integration and enables a high QTZ resolution. The 28nm CMOS protot...
7,VLSI,2024,SAR,0.04,81.8,1.25e+06,9e-05,0.059,A Low-OSR 5th-Order Noise Shaping SAR ADC Using EF-EF-CIFF Structure with PVT-Robust Differential V-T-V Converter,This paper presents a low-OSR 5th-order noise-shaping (NS) SAR (NS-SAR) ADC featuring a differential-Voltage-Time-Voltage (dVTV) converter. It introduces an innovative EF-EF-CIFF structure that ac...
8,ISSCC,2007,"SDCT, SDSC",0.065,82,1.23e+06,0.0031,0.125,A 5th-order CT/DT Multi-Mode ΣΔ Modulator,"A 5th-order CT/DT multi-mode DeltaSigma ADC for the digitisation of baseband signals is presented. For accurate loop characteristics, the design uses DT switched-capacitor OTAs for the second- to ..."
9,ISSCC,2007,SDCT,0.09,82,4e+05,0.00144,0.36,"A 1.2V, 121-Mode Continuous-Time SD Modulator for Wireless Receivers in 90nm CMOS","A reconfigurable CT 5th-order 1b DeltaSigma modulator is presented. The DR/BW is programmable from 85dB@100kHz to 52dB@10MHz in 121 steps. Implemented in a 90nm CMOS process, the 0.36mm 2 IC inclu..."


## 📈 Scatter plot : SNDR vs fsnyq (Walden plot)

In [12]:
import plotly.graph_objects as go
import numpy as np

# --- 1. Préparation ---
def format_unit(val, unit_base):
    if val is None or np.isnan(val): return "N/A"
    if val >= 1e9: return f"{val/1e9:.2f} G{unit_base}"
    if val >= 1e6: return f"{val/1e6:.2f} M{unit_base}"
    if val >= 1e3: return f"{val/1e3:.2f} k{unit_base}"
    if val >= 1:   return f"{val:.2f} {unit_base}"
    if val >= 1e-3: return f"{val*1e3:.2f} m{unit_base}"
    if val >= 1e-6: return f"{val*1e6:.2f} u{unit_base}"
    return f"{val*1e9:.2f} n{unit_base}"

for df in [df_all, results]:
    df['P_fmt'] = df['P [W]'].apply(lambda x: format_unit(x, 'W'))
    df['fs_fmt'] = df['fsnyq [Hz]'].apply(lambda x: format_unit(x, 'Hz'))

# --- 2. Figure ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_all['fsnyq [Hz]'], y=df_all['SNDR_plot [dB]'],
    mode='markers', name='Tous les ADC',
    marker=dict(color='#cccccc', size=6),
    text=df_all['TITLE'],
    customdata=np.stack((df_all['fs_fmt'], df_all['P_fmt']), axis=-1),
    hovertemplate="<b>%{text}</b><br>fsnyq: %{customdata[0]}<br>SNDR: %{y} dB<br>P: %{customdata[1]}<extra></extra>"
))

colors = {'ISSCC': '#e05252', 'VLSI': '#4a90d9'}
for src, grp in results.dropna(subset=['fsnyq [Hz]', 'SNDR_plot [dB]']).groupby('SOURCE'):
    fig.add_trace(go.Scatter(
        x=grp['fsnyq [Hz]'], y=grp['SNDR_plot [dB]'],
        mode='markers', name=f'{src}',
        marker=dict(color=colors.get(src, 'green'), size=10, line=dict(color='black', width=0.5)),
        text=grp['TITLE'],
        customdata=np.stack((grp['fs_fmt'], grp['P_fmt']), axis=-1),
        hovertemplate="<b>%{text}</b><br>fsnyq: %{customdata[0]}<br>SNDR: %{y} dB<br>P: %{customdata[1]}<extra></extra>"
    ))
fig.add_trace(go.Scatter(
    x=[1e6], y=[98],
    mode='markers', name='Mon point',
    marker=dict(symbol='star', color='yellow', size=20, line=dict(color='black', width=1)),
    text=['Mon point de test'],
    customdata=[['1.00 MHz', '3.00 mW']],
    hovertemplate="<b>%{text}</b><br>fsnyq: %{customdata[0]}<br>SNDR: %{y} dB<br>P: %{customdata[1]}<extra></extra>"
))

fig.update_layout(
    title='ADC Survey — SNDR vs fsnyq',
    width=1400, height=850,
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(
        type='log', tickmode='array',
        tickvals=[10**i for i in range(2, 11)],
        ticktext=['100', '1k', '10k', '100k', '1M', '10M', '100M', '1G', '10G'],
        gridcolor='#e0e0e0', showline=True, linecolor='black',
        title='fsnyq [Hz]'
    ),
    yaxis=dict(gridcolor='#e0e0e0', showline=True, linecolor='black', title='SNDR [dB]'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="closest"
)

# --- 3. Tout dans un seul bloc HTML ---
plot_html = fig.to_html(full_html=False, include_plotlyjs='cdn')

full_html = f"""
<div style="font-family:sans-serif;">
  <!-- Barre de copie -->
<div style="display:flex; align-items:center; gap:10px; padding:8px 12px;
              background:#f0f4fa; border:1px solid #c0cce0; border-radius:6px;
              margin-bottom:8px; width:1400px; box-sizing:border-box;">
    <span style="font-weight:600; white-space:nowrap;">📋 Titre :</span>
    <input id="title-box" type="text" readonly value="— survolez un point —"
           style="flex:1; padding:5px 8px; border:1px solid #ccc; border-radius:4px;
                  font-size:13px; background:white;" />
    <button id="copy-btn" onclick="
        var val = document.getElementById('title-box').value;
        navigator.clipboard.writeText(val).then(function() {{
          document.getElementById('copy-btn').textContent = '✅ Copié !';
          setTimeout(function() {{
            document.getElementById('copy-btn').textContent = '📄 Copier';
          }}, 1500);
        }});
      "
      style="padding:5px 14px; background:#4a90d9; color:white; border:none;
             border-radius:4px; cursor:pointer; font-size:13px;">
      📄 Copier
    </button>
  </div>

  <!-- Graphe Plotly -->
  {plot_html}

  <!-- Event hover — même document, ça marche -->
  <script>
    (function attach() {{
      var divs = document.querySelectorAll('.plotly-graph-div');
      var gd = divs[divs.length - 1];
      if (!gd || typeof gd.on !== 'function') {{
        setTimeout(attach, 300);
        return;
      }}
      gd.on('plotly_click', function(data) {{
        if (data.points && data.points.length > 0) {{
          document.getElementById('title-box').value = data.points[0].text || '(sans titre)';
          document.getElementById('copy-btn').textContent = '📄 Copier';
        }}
      }});
    }})();
  </script>
</div>
"""

from IPython.display import display, HTML
display(HTML(full_html))